In [ ]:
import os
import zipfile

# 1. Dézippage du fichier
zip_path = '/content/datasets.zip'  # Assure-toi que le nom est exact
extract_path = '/content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dézippage terminé.")
else:
    print("❌ Erreur : Le fichier datasets.zip est introuvable dans /content/")


✅ Dézippage terminé.

--- Vérification des fichiers ---
Dataset: arxiv_fine           | ❌ Dossier introuvable
Dataset: go_emotion           | ❌ Dossier introuvable
Dataset: massive_intent       | ❌ Dossier introuvable
Dataset: massive_scenario     | ❌ Dossier introuvable
Dataset: mtop_intent          | ❌ Dossier introuvable


In [ ]:
import os
import shutil

# Chemins
base_path = '/content/dataset'
sub_folder = os.path.join(base_path, 'datasets')

# 1. Si le sous-dossier 'datasets' existe, on remonte tout d'un niveau
if os.path.exists(sub_folder):
    for item in os.listdir(sub_folder):
        s = os.path.join(sub_folder, item)
        d = os.path.join(base_path, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d) # Nettoyage si existe déjà
            shutil.move(s, d)
    os.rmdir(sub_folder)
    print("✅ Structure corrigée : les datasets sont maintenant à la racine de /content/dataset/")

# 2. Vérification finale
expected_datasets = ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']
print("\n--- État final des fichiers ---")

for data in expected_datasets:
    folder_path = os.path.join(base_path, data)
    if os.path.exists(folder_path):
        files = os.listdir(folder_path)
        has_small = 'small.jsonl' in files
        has_large = 'large.jsonl' in files

        if has_small and has_large:
            print(f"Dataset: {data:<20} | ✅ OK (small & large présents)")
        else:
            print(f"Dataset: {data:<20} | ⚠️  Fichiers manquants : {[f for f in ['small.jsonl', 'large.jsonl'] if f not in files]}")
    else:
        print(f"Dataset: {data:<20} | ❌ Dossier introuvable")

✅ Structure corrigée : les datasets sont maintenant à la racine de /content/dataset/

--- État final des fichiers ---
Dataset: arxiv_fine           | ✅ OK (small & large présents)
Dataset: go_emotion           | ✅ OK (small & large présents)
Dataset: massive_intent       | ✅ OK (small & large présents)
Dataset: massive_scenario     | ✅ OK (small & large présents)
Dataset: mtop_intent          | ✅ OK (small & large présents)


In [ ]:
import os

# 1. Création du dossier pour les labels générés
output_dir = '/content/generated_labels'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"✅ Dossier créé : {output_dir}")

✅ Dossier créé : /content/generated_labels


In [ ]:

# 2. On écrit le script select_part_labels.py dans le Colab
with open('select_part_labels.py', 'w') as f:
    f.write('''
import json
import random
import os

def find_sorted_folders(directory):
    folders = []
    for entry in os.scandir(directory):
        if entry.is_dir():
            folders.append(entry.name)
    folders.sort()
    return folders

def load_dataset(data_path, data):
    data_file = os.path.join(data_path, data, "small.jsonl")
    with open(data_file,'r') as f:
        data_list = []
        for line in f:
            json_object = json.loads(line)
            data_list.append(json_object)
    return data_list

def get_label_list(data_list):
    label_list = []
    for data in data_list:
        if data["label"] not in label_list:
            label_list.append(data["label"])
    return label_list

def main():
    data_path = "./dataset/"
    total_chosen_labels = dict()
    folders = find_sorted_folders(data_path)

    for data in folders:
        if data not in ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']:
            continue
        data_list = load_dataset(data_path, data)
        data_labels = get_label_list(data_list)
        choose_num = int(0.2 * len(data_labels))
        # Utilisation de random.sample pour éviter les doublons dans la sélection
        total_chosen_labels[data] = random.sample(data_labels, k=choose_num)
        print(f"Dataset: {data:<20} | Labels totaux: {len(data_labels):<3} | Sélection (20%): {choose_num}")

    with open("./generated_labels/chosen_labels.json", 'w') as f:
        json.dump(total_chosen_labels, f, indent=2)
    print("\\n✅ Fichier ./generated_labels/chosen_labels.json généré avec succès.")

if __name__ == "__main__":
    main()
''')

# 3. Exécution du script
!python select_part_labels.py

Dataset: arxiv_fine           | Labels totaux: 93  | Sélection (20%): 18
Dataset: go_emotion           | Labels totaux: 27  | Sélection (20%): 5
Dataset: massive_intent       | Labels totaux: 59  | Sélection (20%): 11
Dataset: massive_scenario     | Labels totaux: 18  | Sélection (20%): 3
Dataset: mtop_intent          | Labels totaux: 102 | Sélection (20%): 20

✅ Fichier ./generated_labels/chosen_labels.json généré avec succès.


In [ ]:
# 1. Installation de zstd et d'Ollama
print("Installation de zstd et Ollama...")
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Lancement du serveur Ollama en arrière-plan
import subprocess
import time
import os

print("Démarrage du serveur Ollama...")
# On redirige la sortie vers un fichier log pour éviter de bloquer la cellule
with open("ollama_server.log", "w") as f:
    process = subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=f)

# On attend que le serveur soit prêt
time.sleep(5)

# 3. Téléchargement du modèle Llama 3.1 (8B)
print("Téléchargement du modèle Llama 3.1:8b (environ 4.7 Go)...")
!ollama pull llama3.1:8b

print("\n✅ Ollama est prêt et Llama 3.1 est installé !")

Installation de zstd et Ollama...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,432 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,847 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Get:13 https://ppa.launc

In [ ]:
import os
import subprocess
import time
import requests

# 1. Tuer toute instance précédente pour repartir à zéro
!pkill ollama
time.sleep(2)

# 2. Lancer le serveur en arrière-plan
print("Démarrage du serveur Ollama...")
with open("ollama_server.log", "w") as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=f)

# 3. Attendre que le port 11434 soit ouvert
print("Attente du serveur (max 30s)...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Serveur Ollama en ligne.")
            break
    except:
        time.sleep(1)
else:
    print("❌ Le serveur met trop de temps à démarrer.")

# 4. Forcer le chargement de Llama 3.1 en mémoire (pour éviter le timeout au test)
print("Pré-chargement de Llama 3.1:8b en mémoire GPU...")
!ollama run llama3.1:8b "Bonjour" # Cette commande force le chargement initial

Démarrage du serveur Ollama...
Attente du serveur (max 30s)...
✅ Serveur Ollama en ligne.
Pré-chargement de Llama 3.1:8b en mémoire GPU...
Bonjour ! Comment puis-je vous aider aujourd'hui ?



In [ ]:
import requests

def test_llama_local():
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "llama3.1:8b",
        "prompt": "Réponds exactement par le mot 'OK' si tu fonctionnes.",
        "stream": False
    }
    try:
        # On augmente le timeout à 60 secondes pour le premier chargement
        response = requests.post(url, json=payload, timeout=60)
        if response.status_code == 200:
            result = response.json().get('response', '').strip()
            print(f"Réponse du LLM : {result}")
            if "OK" in result.upper():
                print("✅ TEST RÉUSSI : Connexion établie avec succès !")
            else:
                print("⚠️ Le serveur répond mais la réponse est inattendue.")
        else:
            print(f"❌ Erreur de réponse du serveur (Code: {response.status_code})")
    except Exception as e:
        print(f"❌ Erreur de connexion : {e}")

test_llama_local()

Réponse du LLM : OK
✅ TEST RÉUSSI : Connexion établie avec succès !


## **label_generation**

- Génération : Pour chaque dataset, Llama 3.1 va lire les phrases par groupes de 15 et proposer des noms de labels.

- Fusion (Merge) : À la fin de chaque dataset, il va prendre la liste globale et fusionner les synonymes pour réduire le nombre de clusters (ton point bloquant précédent).

- Sauvegarde : Les fichiers seront créés dans le dossier /content/generated_labels/.

In [ ]:
import subprocess

datasets = ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']

# Paramètres :
# test_num=20 signifie qu'on regarde 20 batches de 15 phrases (300 phrases par dataset)
# Tu peux augmenter ce chiffre si tu veux plus de précision, mais cela prendra plus de temps.
test_num = 20

for ds in datasets:
    print(f"\n" + "="*50)
    print(f"🚀 Lancement de la génération pour : {ds}")
    print("="*50)

    # Construction de la commande
    cmd = [
        "python", "label_generation.py",
        "--data", ds,
        "--test_num", str(test_num),
        "--chunk_size", "15"
    ]

    # Exécution et affichage en temps réel
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(result.stdout)
        print(f"✅ Terminé avec succès pour {ds}")
    else:
        print(f"❌ Erreur pour {ds} :")
        print(result.stderr)

print("\n\n" + "!"*50)
print("TOUS LES DATASETS ONT ÉTÉ TRAITÉS !")
print("!"*50)


🚀 Lancement de la génération pour : arxiv_fine
--- Starting Label Generation for arxiv_fine ---
use_large:  False
Use dataset ./dataset/arxiv_fine/small.jsonl
Length of dataset: 3674
Ground Truth cluster num: 93
JSON file './generated_labels/arxiv_fine_small_true_labels.json' written.
Batch 1: 23 total labels
Batch 2: 40 total labels
Batch 3: 46 total labels
Batch 4: 46 total labels
Batch 5: 63 total labels
Batch 6: 75 total labels
Batch 8: 89 total labels
Batch 9: 92 total labels
Batch 10: 93 total labels
Batch 11: 104 total labels
Batch 12: 112 total labels
Batch 13: 116 total labels
Batch 14: 116 total labels
Batch 15: 116 total labels
Batch 16: 117 total labels
Batch 17: 118 total labels
Batch 18: 132 total labels
Batch 19: 149 total labels
Batch 20: 164 total labels
Total labels given by LLM (before merge): 164
JSON file './generated_labels/arxiv_fine_small_llm_generated_labels_before_merge.json' written.
Merging similar labels...
JSON file './generated_labels/arxiv_fine_small_ll

## **given_label_classification**

In [ ]:
datasets = ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']

for ds in datasets:
    print(f"\n" + "="*60)
    print(f"🚀 PHASE 1 : Classification TOTALE de {ds}")
    print("="*60)

    # On traite 100% du dataset
    !python given_label_classification.py --data $ds --test_num 0

    0


🚀 PHASE 1 : Classification TOTALE de arxiv_fine
Loading dataset: ./dataset/arxiv_fine/small.jsonl
Classifying 3674 samples for arxiv_fine...
100% 3674/3674 [42:05<00:00,  1.45it/s]

✅ Results saved to ./generated_labels/arxiv_fine_small_find_labels.json

🚀 PHASE 1 : Classification TOTALE de go_emotion
Loading dataset: ./dataset/go_emotion/small.jsonl
Classifying 5940 samples for go_emotion...
100% 5940/5940 [1:00:12<00:00,  1.64it/s]

✅ Results saved to ./generated_labels/go_emotion_small_find_labels.json

🚀 PHASE 1 : Classification TOTALE de massive_intent
Loading dataset: ./dataset/massive_intent/small.jsonl
Classifying 2974 samples for massive_intent...
100% 2974/2974 [29:29<00:00,  1.68it/s]

✅ Results saved to ./generated_labels/massive_intent_small_find_labels.json

🚀 PHASE 1 : Classification TOTALE de massive_scenario
Loading dataset: ./dataset/massive_scenario/small.jsonl
Classifying 2974 samples for massive_scenario...
100% 2974/2974 [29:15<00:00,  1.69it/s]

✅ Results saved 

## **evaluate**

In [ ]:
datasets = ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']

for ds in datasets:
    # On met bien --test_num 0 pour évaluer TOUT le dataset
    !python evaluate.py --data $ds --test_num 0


--- Results for arxiv_fine (3674 samples) ---
NMI: 0.3513
ARI: 0.0393
ACC: 0.1331
Clusters prédits: 133 | Clusters réels: 93

--- Results for go_emotion (5940 samples) ---
NMI: 0.1946
ARI: 0.0583
ACC: 0.1860
Clusters prédits: 71 | Clusters réels: 27

--- Results for massive_intent (2974 samples) ---
NMI: 0.5661
ARI: 0.2951
ACC: 0.3850
Clusters prédits: 84 | Clusters réels: 59

--- Results for massive_scenario (2974 samples) ---
NMI: 0.5332
ARI: 0.2976
ACC: 0.3958
Clusters prédits: 80 | Clusters réels: 18

--- Results for mtop_intent (4386 samples) ---
NMI: 0.5830
ARI: 0.3261
ACC: 0.3931
Clusters prédits: 88 | Clusters réels: 102


##**evaluate avec 500 phrases**

In [ ]:
datasets = ['arxiv_fine', 'go_emotion', 'massive_intent', 'massive_scenario', 'mtop_intent']

for ds in datasets:
    # On passe bien --test_num 500 pour correspondre à ta classification
    !python evaluate.py --data $ds --test_num 500


--- Results for arxiv_fine (500 samples) ---
NMI: 0.5094
ARI: 0.2451
ACC: 0.3860
Clusters prédits: 49 | Clusters réels dans l'échantillon: 13

--- Results for go_emotion (500 samples) ---
NMI: 0.3375
ARI: 0.0646
ACC: 0.2440
Clusters prédits: 43 | Clusters réels dans l'échantillon: 27

--- Results for massive_intent (500 samples) ---
NMI: 0.6429
ARI: 0.4963
ACC: 0.5260
Clusters prédits: 27 | Clusters réels dans l'échantillon: 31

--- Results for massive_scenario (500 samples) ---
NMI: 0.6023
ARI: 0.4331
ACC: 0.5360
Clusters prédits: 35 | Clusters réels dans l'échantillon: 11

--- Results for mtop_intent (500 samples) ---
NMI: 0.7176
ARI: 0.5283
ACC: 0.5900
Clusters prédits: 35 | Clusters réels dans l'échantillon: 60
